# 13 — Regiões anatômicas das ROIs mais importantes (TEA)

Este notebook lê os dois modelos mais bem classificados no caderno 12 e suas importâncias por ROI calculadas no caderno 11. Para cada ROI entre as mais importantes, calcula a sobreposição com o AAL e informa as regiões anatômicas correspondentes.

> O CC200 é uma parcellação funcional e não fornece nomes anatômicos únicos para todas as ROIs. A correspondência com AAL é aproximada: uma ROI pode cobrir mais de uma região, ou não ter sobreposição suficiente. A tabela mantém os percentuais de sobreposição para tornar isso visível.

**Pré-requisitos:** executar os cadernos 11 e 12. O atlas CC200 deve estar em `Data/raw/cc200_roi_atlas.nii.gz`. O atlas AAL é baixado pelo Nilearn se ainda não estiver em cache.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import re
import numpy as np
import pandas as pd
import nibabel as nib
from nilearn import datasets, image

raiz = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'src' / 'config.py').is_file())
from src.config import PATHS

arquivo_modelos = raiz / 'Results' / '12_atualizados' / '12_tabela_final.csv'
if not arquivo_modelos.exists():
    arquivo_modelos = raiz / 'Results' / '12_tabela_final.csv'
arquivo_importancia = raiz / 'Results' / '11_importancia_roi.csv'
arquivo_cc200 = Path(PATHS['raw']) / 'cc200_roi_atlas.nii.gz'
if not arquivo_cc200.exists():
    arquivo_cc200.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve('https://fcp-indi.s3.amazonaws.com/data/Projects/ABIDE_Initiative/Resources/cc200_roi_atlas.nii.gz', arquivo_cc200)
TOP_N = 15  # ROIs mais importantes por modelo; ajuste se desejar

for arq in [arquivo_modelos, arquivo_importancia, arquivo_cc200]:
    if not arq.exists():
        raise FileNotFoundError(f'Arquivo não encontrado: {arq}. Rode os cadernos 11/12 e confira o atlas CC200.')

resultado = pd.read_csv(arquivo_modelos, index_col=0)
resultado = resultado[~resultado['família'].astype(str).str.startswith('Referência')]
modelos_top2 = resultado.sort_values('AUC média (folds)', ascending=False).head(2).index.astype(str).tolist()
imp = pd.read_csv(arquivo_importancia)
print('Dois melhores modelos:', modelos_top2)
display(resultado.loc[modelos_top2, ['família', 'AUC média (folds)', 'AUC OOF']])
display(imp.head())

In [ ]:
# Busca da tabela anatômica por sobreposição espacial CC200 × AAL
cc_img = nib.load(str(arquivo_cc200))
cc_data = np.asarray(cc_img.dataobj).squeeze()
if cc_data.ndim != 3:
    raise ValueError(f'Atlas CC200 deveria ser 3D; shape recebido: {cc_data.shape}')

aal = datasets.fetch_atlas_aal(version='SPM12')
aal_img = image.load_img(aal.maps)
aal_on_cc = image.resample_to_img(aal_img, cc_img, interpolation='nearest')
aal_data = np.asarray(aal_on_cc.dataobj).squeeze().astype(int)
aal_labels = list(aal.labels)
# Em algumas versões, o índice zero representa o fundo.
if len(aal_labels) and str(aal_labels[0]).lower() in {'background', '0', 'none'}:
    aal_labels = aal_labels[1:]

def nome_roi_cc200(rotulo):
    # As matrizes do projeto usam IDs CC200 começando em 1.
    s = str(rotulo)
    m = re.search(r'(\d+)', s)
    if not m:
        raise ValueError(f'Não consegui extrair o ID CC200 de {rotulo!r}')
    return int(m.group(1))

def mapear_roi(roi_texto):
    roi_id = nome_roi_cc200(roi_texto)
    mascara = np.isclose(cc_data, roi_id)
    n_vox = int(mascara.sum())
    if n_vox == 0:
        return {'roi_id_cc200': roi_id, 'regioes_aal': 'ROI não encontrada no atlas', 'sobreposicao_aal_pct': ''}
    valores, contagens = np.unique(aal_data[mascara], return_counts=True)
    pares = [(int(v), int(n)) for v, n in zip(valores, contagens) if v > 0]
    pares.sort(key=lambda x: x[1], reverse=True)
    nomes = []
    for idx, n in pares:
        # AAL usa índices 1..N; lista de nomes varia entre versões do Nilearn.
        nome = str(aal_labels[idx - 1]) if idx - 1 < len(aal_labels) else f'AAL_{idx}'
        nomes.append(f'{nome} ({100*n/n_vox:.1f}%)')
    return {'roi_id_cc200': roi_id, 'regioes_aal': '; '.join(nomes) if nomes else 'Sem correspondência AAL',
            'sobreposicao_aal_pct': round(100*sum(n for _, n in pares)/n_vox, 1)}

linhas = []
for modelo in modelos_top2:
    coluna_importancia = 'queda_auc_permutacao' if 'queda_auc_permutacao' in imp.columns and imp.loc[imp['modelo'].astype(str).eq(modelo), 'queda_auc_permutacao'].notna().any() else 'importancia_linear'
    sub = imp[imp['modelo'].astype(str).eq(modelo)].sort_values(coluna_importancia, ascending=False).head(TOP_N)
    for _, row in sub.iterrows():
        m = mapear_roi(row['roi'])
        linhas.append({'modelo': modelo, 'roi': row['roi'], 'criterio_importancia': coluna_importancia,
                      'importancia': row[coluna_importancia], **m})
tabela = pd.DataFrame(linhas)
display(tabela)
saida = raiz / 'Results' / '13_mapeamento_rois_cc200.csv'
tabela.to_csv(saida, index=False)
print('Tabela salva em:', saida)